# Prepare dynamics data

Inspect a raw motor dataset and select the acceleration channels used by the RNN.
The trainer can also read the raw CSVs directly with `--input-format motors --output-format plant`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/model/rnn_loader.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from model.training.train import read_csv, prepare_sequences

MODE = "z_psi"
DATA_DIR = PROJECT_ROOT / "data/generated_data"
INPUT_FILE = DATA_DIR / "db_regime6_helico_input.csv"
OUTPUT_FILE = DATA_DIR / "db_regime6_helico_output.csv"
MAX_TRAJECTORIES = 32  # Use None for the full dataset.

inputs, outputs = prepare_sequences(
    read_csv(INPUT_FILE, MAX_TRAJECTORIES),
    read_csv(OUTPUT_FILE, MAX_TRAJECTORIES),
    MODE,
    input_format="motors",
    output_format="plant",
)
print(f"{len(inputs)} trajectories, {inputs.shape[1]} samples each")


The same motor conversion and output ordering are used here and during training.
For scalar z data, all four motor commands must be equal. Set `MODE = "z"` and select that dataset.


In [ ]:
input_names = ["thrust", "roll", "pitch", "yaw"][:inputs.shape[-1]]
output_names = ["az", "wdotz"][:outputs.shape[-1]]

for names, values in [(input_names, inputs), (output_names, outputs)]:
    for index, name in enumerate(names):
        signal = values[..., index]
        print(f"{name}: min {signal.min():.4g}, max {signal.max():.4g}, std {signal.std():.4g}")

trajectory = 0
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(inputs[trajectory], label=input_names)
axes[1].plot(outputs[trajectory], label=output_names)
axes[0].set_ylabel("Control")
axes[1].set_ylabel("Acceleration")
axes[1].set_xlabel("Sample")
for axis in axes:
    axis.legend()
    axis.grid(alpha=0.3)
fig.tight_layout()


Set `SAVE = True` to write the selected channels. The source CSVs are preserved.
The saved files use the trainer defaults: controls as inputs and selected accelerations as outputs.


In [ ]:
SAVE = False
OUTPUT_DIR = PROJECT_ROOT / "data/processed" / MODE

if SAVE:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    np.savetxt(OUTPUT_DIR / "inputs.csv", inputs.reshape(len(inputs), -1), delimiter=",")
    np.savetxt(OUTPUT_DIR / "outputs.csv", outputs.reshape(len(outputs), -1), delimiter=",")
    print(OUTPUT_DIR)
